# Sultan Hassan SDXL LoRA Training — FIXED (Local/Colab Edition)
This notebook contains the exact fixed setup using xformers, the v0.30.3 training script, and the fp16-fix VAE to guarantee it runs flawlessly on a local GPU or Colab without memory crashes or black images.

In [ ]:
# Step 1: Install required dependencies (No bitsandbytes required)
!pip install -q diffusers==0.30.3 transformers accelerate peft datasets wandb xformers

In [ ]:
# Step 2: Unzip the Dataset locally
import os
import zipfile

if not os.path.exists('dataset'):
    os.makedirs('dataset')

if os.path.exists('data_final.zip'):
    with zipfile.ZipFile('data_final.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print('Dataset extracted! Found', len(os.listdir('dataset')), 'images.')
else:
    print('Make sure data_final.zip is in this folder!')


In [ ]:
# Step 3: Download the official v0.30.3 training script (No patches needed)
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.30.3/examples/dreambooth/train_dreambooth_lora_sdxl.py
print("Stable script downloaded.")


In [ ]:
# Step 4: Configure accelerate non-interactively
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")
print("accelerate config written.")


In [ ]:
# Step 5: Start Training! (Uses xformers and fp16-fix VAE)
!accelerate launch --mixed_precision="fp16" train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --instance_data_dir="dataset" \
  --instance_prompt="sltnhsn" \
  --output_dir="sultan_hassan_sdxl_lora" \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --checkpointing_steps=250 \
  --seed=42 \
  --gradient_checkpointing \
  --enable_xformers_memory_efficient_attention


In [ ]:
# Step 6: Generate beautiful test images right here!
from diffusers import DiffusionPipeline, AutoencoderKL
import torch
from IPython.display import display

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

pipe.load_lora_weights("sultan_hassan_sdxl_lora")

prompt = "A beautiful professional photograph of sltnhsn at sunset, highly detailed, dramatic lighting"
image = pipe(prompt, num_inference_steps=30).images[0]

image.save("sultan_hassan_test.png")
display(image)
